# Insurance Claim Volume Forecasting

## Project Overview

Forecasts **daily insurance claim volume** over a 14-day horizon. Synthetic data spans 2 years with weekly patterns, seasonal peaks, and catastrophe events.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily claim counts, predict the next 14 days. Insurance claim volume forecasts enable staffing optimization, reserve adequacy assessment, and reinsurance planning.

## Why This Project Matters

Insurance companies must maintain adequate staffing and reserves to handle claims. Understaffing delays payouts and harms customers; overstaffing wastes resources. Catastrophe events (storms, floods) create sudden surges. Accurate forecasts enable proactive resource allocation.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily claim counts
- Weekly pattern (lower on weekends)
- Seasonal variation (winter driving hazards, summer storms)
- Catastrophe events (surges)
- Gradual growth trend (portfolio growth)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `claims` | Daily insurance claim count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'claims'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Base level with growth
base = 80 + 0.02 * t

# Weekly pattern
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4:
        weekly[i] = 10  # weekday reporting
    elif dow == 5:
        weekly[i] = -15
    else:
        weekly[i] = -25  # Sunday lowest

# Seasonal: winter ice/storms + summer storms
seasonal = 8 * np.cos(2 * np.pi * (t - 15) / 365) + 5 * np.sin(2 * np.pi * (t - 180) / 365)

# Catastrophe events
cat_events = np.zeros(N_POINTS)
for s in range(0, N_POINTS, 180):
    cat_day = min(s + np.random.randint(60, 150), N_POINTS - 7)
    surge = np.random.uniform(40, 100)
    for j in range(min(7, N_POINTS - cat_day)):
        cat_events[cat_day + j] = surge * np.exp(-0.3 * j)

noise = np.random.normal(0, 5, N_POINTS)
claims = base + weekly + seasonal + cat_events + noise
claims = np.maximum(claims, 5).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'claims': claims})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,claims
0,2022-01-01,73
1,2022-01-02,62
2,2022-01-03,98
3,2022-01-04,95
4,2022-01-05,95


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('claims Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("claims Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

claims Statistics:
count    730.000000
mean      90.502740
std       19.846271
min       38.000000
25%       80.000000
50%       93.000000
75%      101.000000
max      213.000000
Name: claims, dtype: float64

CV: 0.219
Skewness: 0.832


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       41.7 | RMSE:       51.7 | MAPE: 29.90%
Seasonal Naive (7)             | MAE:       20.5 | RMSE:       30.9 | MAPE: 13.71%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared  R-Squared       RMSE  Time Taken
Model                                                                          
GaussianProcessRegressor           460.020319 -34.309255  86.407674    0.050025
KernelRidge                        451.635480 -33.664268  85.614840    0.026276
MLPRegressor                        83.423526  -5.340271  36.615246    0.398057
OrthogonalMatchingPursuit           36.405417  -1.723494  23.997788    0.008192
DummyRegressor                      30.092543  -1.237888  21.753403    0.007906
QuantileRegressor                   25.683846  -0.898757  20.037465    0.058155
NuSVR                               13.846156   0.011834  14.455162    0.020149
SVR                                 13.332939   0.051312  14.163469    0.029557
KNeighborsRegressor                 10.260856   0.287626  12.273316    0.011790
DecisionTreeRegressor               10.107685   0.299409  12.171395    0.013205


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        6.1 | RMSE:        7.0 | MAPE:  5.66%
FLAML Test (catboost)          | MAE:       14.8 | RMSE:       24.7 | MAPE:  9.77%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       21.8 | RMSE:       30.9 | MAPE: 15.27%
SF AutoETS                     | MAE:       19.5 | RMSE:       29.4 | MAPE: 13.20%
SF SeasonalNaive               | MAE:       20.9 | RMSE:       32.1 | MAPE: 13.93%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  6.072535  6.961285  5.660692
FLAML Test (catboost) 14.760866 24.698954  9.774709
           SF AutoETS 19.522142 29.378721 13.196471
   Seasonal Naive (7) 20.500000 30.851025 13.712313
     SF SeasonalNaive 20.857143 32.124757 13.927025
         SF AutoARIMA 21.806942 30.921308 15.273665
   Naive (Last Value) 41.714286 51.745255 29.896102

Top 3:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  6.072535  6.961285  5.660692
FLAML Test (catboost) 14.760866 24.698954  9.774709
           SF AutoETS 19.522142 29.378721 13.196471


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 14.09, Std: 20.29


## Interpretation and Business Insight

- Claim volume drops sharply on weekends (reporting behavior)
- Winter and summer have higher claims (ice hazards + storms)
- Catastrophe events create multi-day claim surges
- Portfolio growth creates a gradual upward trend
- Staffing should be highest on Mondays (weekend + Monday claims)

## Limitations

1. Synthetic — real claims depend on policy portfolio, geography, weather
2. No claim severity data (count only, not dollar amounts)
3. No weather features (key driver for property claims)
4. Single line of business — real insurers have multiple
5. No claim development patterns (IBNR)

## How to Improve This Project

1. Add weather data for property/auto claim prediction
2. Include claim severity for reserve estimation
3. Model by line of business (auto, property, health)
4. Add IBNR (incurred but not reported) modeling
5. Use spatial data for regional claim hotspots

## Production Considerations

- Daily claim volume forecasting for call center staffing
- Catastrophe event detection and escalation
- Reserve adequacy monitoring
- Reinsurance trigger monitoring

## Common Mistakes

1. Ignoring weekend vs weekday reporting patterns
2. Not separating attritional claims from catastrophe claims
3. Using claim count without severity for financial reserves
4. Treating all lines of business the same
5. Not accounting for policy portfolio growth in trend

## Mini Challenge / Exercises

1. Detect catastrophe events from the claim series
2. Build separate weekday and weekend models
3. Create a claim surge probability forecast
4. Compare winter vs summer claim patterns

## Final Summary / Key Takeaways

1. Insurance claim volume has strong weekly and seasonal patterns
2. Catastrophe events are the primary source of forecast error
3. Staffing optimization is the highest-value application
4. Real deployment needs weather integration and severity modeling
5. Separating attritional from catastrophe claims improves forecasts

In [18]:
import json
metrics = {
    'project': 'Insurance Claim Volume Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Insurance Claim Volume Forecasting — Complete ===")

Metrics saved.

=== Insurance Claim Volume Forecasting — Complete ===
